# Sentinel — Phase 2.1 fine-tuning (Colab GPU)

Runs the **real** detector fine-tune on a free Colab GPU — the job that the local CPU can only smoke-test. Trains both D1 bake-off candidates (YOLO12 and RF-DETR) **on the same data**, evaluates them, exports the winner, and zips the weights for download back into the Sentinel repo.

**Before running:** Runtime → Change runtime type → **GPU (T4)**.

See `training/README.md` and `DECISIONS.md` D1 for the reasoning behind every choice here.

## 1. Environment

In [ ]:
!pip -q install ultralytics "rfdetr[train,loggers]" roboflow sahi opencv-python pyyaml
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime > Change runtime type > GPU'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Dataset

Two options. Pick one.

**Option A (recommended for the real pilot) — your own Roboflow dataset.** On the dataset's Roboflow Universe page, click *Download this Dataset* → it gives you an exact snippet. Paste it below. Export **YOLOv8** format for YOLO12, and **COCO** format for RF-DETR. The classes should be your remote-home set: `person`, `vehicle`, `package`, `animal`.

**Option B (zero-config fallback) — Pascal VOC.** A real academic dataset that auto-downloads and covers `person`/`car`/`dog`/`cat` (3 of 4 pilot classes; no `package`). Good for a first real run end-to-end. Set `USE_VOC = True`. RF-DETR needs COCO-format data, which VOC.yaml doesn't provide directly — the next cell converts it automatically (same conversion logic as `training/yolo_to_coco.py`, tested against a real training run in the Sentinel repo).

In [ ]:
USE_VOC = True  # set False and fill in Option A below for the real pilot dataset

if USE_VOC:
    YOLO_DATA = 'VOC.yaml'   # Ultralytics auto-downloads
else:
    # --- Option A: paste your Roboflow snippets here ---
    from roboflow import Roboflow
    rf = Roboflow(api_key='PASTE_YOUR_KEY')   # or os.environ['ROBOFLOW_API_KEY']
    # YOLO format for YOLO12:
    yolo_ds = rf.workspace('WORKSPACE').project('PROJECT').version(1).download('yolov8')
    YOLO_DATA = yolo_ds.location + '/data.yaml'
    # COCO format for RF-DETR (Roboflow exports both directly — no conversion needed):
    coco_ds = rf.workspace('WORKSPACE').project('PROJECT').version(1).download('coco')
    COCO_DIR = coco_ds.location
print('YOLO_DATA =', YOLO_DATA)

## 2b. Convert to COCO format for RF-DETR (VOC path only)

Ultralytics' `VOC.yaml` auto-download only ships YOLO-format labels. This cell converts them to COCO so RF-DETR gets a genuine two-way bake-off on the *same* data, not a skipped comparison. If you're on Option A, Roboflow already exported COCO directly (`COCO_DIR` is set above) and this cell does nothing.

In [ ]:
import json
from pathlib import Path

import cv2
import yaml


def _read_yolo_labels(label_path, img_w, img_h):
    if not label_path.exists():
        return []
    boxes = []
    for line in label_path.read_text().strip().splitlines():
        if not line.strip():
            continue
        parts = line.split()
        class_id = int(parts[0])
        cx, cy, w, h = (float(x) for x in parts[1:5])
        x_min = (cx - w / 2) * img_w
        y_min = (cy - h / 2) * img_h
        boxes.append({'category_id': class_id, 'bbox': [x_min, y_min, w * img_w, h * img_h]})
    return boxes


def convert_split(images_dir, labels_dir, out_dir, class_names):
    out_dir.mkdir(parents=True, exist_ok=True)
    images_meta, annotations, ann_id = [], [], 1
    image_paths = sorted(p for p in images_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    for img_id, img_path in enumerate(image_paths):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        images_meta.append({'id': img_id, 'file_name': img_path.name, 'width': w, 'height': h})
        for box in _read_yolo_labels(labels_dir / (img_path.stem + '.txt'), w, h):
            annotations.append({
                'id': ann_id, 'image_id': img_id, 'category_id': box['category_id'] + 1,
                'bbox': box['bbox'], 'area': box['bbox'][2] * box['bbox'][3], 'iscrowd': 0,
            })
            ann_id += 1
        dest = out_dir / img_path.name
        if not dest.exists():
            dest.write_bytes(img_path.read_bytes())
    categories = [{'id': i + 1, 'name': name, 'supercategory': 'none'} for i, name in enumerate(class_names)]
    (out_dir / '_annotations.coco.json').write_text(
        json.dumps({'images': images_meta, 'annotations': annotations, 'categories': categories})
    )
    return len(images_meta)


def convert_dataset(yolo_data_yaml, out_dir):
    config = yaml.safe_load(Path(yolo_data_yaml).read_text())
    base = Path(config.get('path', Path(yolo_data_yaml).parent))
    class_names = config['names'] if isinstance(config['names'], list) else list(config['names'].values())
    split_map = {'train': 'train', 'val': 'valid', 'test': 'test'}
    counts = {}
    for yolo_split_key, coco_split_name in split_map.items():
        rel = config.get(yolo_split_key)
        if not rel:
            continue
        images_dir = base / rel
        labels_dir = Path(str(images_dir).replace('/images/', '/labels/', 1))
        if not images_dir.exists():
            continue
        counts[coco_split_name] = convert_split(images_dir, labels_dir, Path(out_dir) / coco_split_name, class_names)
    return counts


if USE_VOC:
    from ultralytics.utils import DATASETS_DIR
    from ultralytics.data.utils import check_det_dataset
    check_det_dataset(YOLO_DATA)  # triggers the actual VOC download/cache if not already present
    voc_yaml_path = Path(DATASETS_DIR) / 'VOC.yaml'
    if not voc_yaml_path.exists():
        import ultralytics, shutil as _sh
        voc_yaml_path = Path(ultralytics.__file__).parent / 'cfg' / 'datasets' / 'VOC.yaml'
    COCO_DIR = '/content/voc_coco'
    counts = convert_dataset(voc_yaml_path, COCO_DIR)
    print('Converted VOC -> COCO:', counts)
    # RF-DETR expects a train/valid/test layout; VOC has no dedicated test split, reuse valid
    import shutil
    if not Path(f'{COCO_DIR}/test').exists():
        shutil.copytree(f'{COCO_DIR}/valid', f'{COCO_DIR}/test')
print('COCO_DIR =', COCO_DIR)

## 3. Train YOLO12

From COCO-pretrained `yolo12s.pt` (not the bare architecture). Backbone frozen for the first stretch to avoid catastrophic forgetting on a small custom set, per `training/README.md`.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo12s.pt')
model.train(
    data=YOLO_DATA,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    freeze=10,           # freeze backbone early; see training/README.md
    patience=20,         # early-stop when val plateaus
    project='/content/drive/MyDrive/sentinel_runs',  # Drive-persisted: survives a Colab disconnect
    name='yolo12',
    # CCTV-realistic augmentation on top of defaults:
    hsv_v=0.5, degrees=5, translate=0.1, scale=0.5, fliplr=0.5, mosaic=1.0,
)
print('YOLO12 best weights: /content/drive/MyDrive/sentinel_runs/yolo12/weights/best.pt')

**Before running the cell above:** mount Google Drive so the run survives a disconnect (a real, verified risk — see `ROADMAP.md` Phase 2.1 for what happened the first time this wasn't done):
```python
from google.colab import drive
drive.mount('/content/drive')
```

## 4. Train RF-DETR

Now runs on the same data as YOLO12 (converted above if you're on the VOC path) — a genuine two-way bake-off, not a skipped comparison.

In [ ]:
from rfdetr import RFDETRBase
m = RFDETRBase()
m.train(
    dataset_dir=COCO_DIR,
    epochs=60,
    batch_size=8,
    grad_accum_steps=2,
    lr=1e-4,
    output_dir='/content/drive/MyDrive/sentinel_runs/rfdetr',
)
print('RF-DETR weights in /content/drive/MyDrive/sentinel_runs/rfdetr/')

## 5. Bake-off — held-out mAP

Compare both candidates on the same held-out split. The full bake-off (DECISIONS.md D1 / TESTING.md) also weighs false-alarms-per-camera-per-day and edge latency — those need the live footage and the target edge box, so they happen back in the Sentinel repo, not here.

In [ ]:
yolo_metrics = YOLO('/content/drive/MyDrive/sentinel_runs/yolo12/weights/best.pt').val(data=YOLO_DATA)
print('YOLO12   mAP50:', round(yolo_metrics.box.map50, 4), ' mAP50-95:', round(yolo_metrics.box.map, 4))

rfdetr_eval = RFDETRBase(pretrain_weights='/content/drive/MyDrive/sentinel_runs/rfdetr/checkpoint_best_regular.pth')
rfdetr_metrics = rfdetr_eval.eval(dataset_dir=COCO_DIR)
print('RF-DETR  metrics:', rfdetr_metrics)

## 6. Export the winner

ONNX is the portable edge format; TensorRT export happens on the actual Jetson (it needs that hardware). The `.pt` is what `edge/detector.py ModelDetector` loads directly.

In [ ]:
YOLO('/content/drive/MyDrive/sentinel_runs/yolo12/weights/best.pt').export(format='onnx')
print('Exported ONNX next to best.pt')

## 7. Use the weights

Weights are already in Google Drive (`sentinel_runs/`), not just Colab's temp disk — download `best.pt` from Drive whenever convenient, drop it into `data/models/` in the repo, and run:
```
SENTINEL_DETECTOR_BACKEND=model SENTINEL_DETECTOR_MODEL_PATH=./data/models/best.pt python -m edge.main
```